# Visualize attention pattern in few-shot prompts

## Loading packages

In [5]:
import circuitsvis
import circuitsvis.attention as cv_attn
import sys
from pathlib import Path
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
from tqdm import tqdm
from typing import List, Dict, Set
import transformers
from transformers import AutoTokenizer, AutoModelForCausalLM

try:
    import transformer_lens
    from transformer_lens import HookedTransformer, utilities, patching
except ImportError:
    print("Installing transformer-lens for mechanistic interpretability...")
    !pip install transformer-lens -q
    import transformer_lens
    from transformer_lens import HookedTransformer, utilities, patching

# Device configuration
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")

# Load utils
REPO_ROOT = Path(r"C:\Users\nguye\repro-shared-lexical-task")
sys.path.insert(0, str(REPO_ROOT / "src"))
from utils.build_prompts import _load_dataset, filter_token_pairs_by_length, create_few_shot_prompts

Using device: cpu


In [2]:
# from huggingface_hub import login
# login()

# Llama-3.1-8B-Instruct
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.1-8B-Instruct", device_map="auto")

# Llama-3.2-1B-Instruct
# tokenizer = AutoTokenizer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
# model = AutoModelForCausalLM.from_pretrained("meta-llama/Llama-3.2-1B-Instruct", device_map="auto")

model = HookedTransformer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")

C:\Users\nguye\AppData\Local\Temp\ipykernel_27528\3652073285.py:12: DeprecationWarning: HookedTransformer.from_pretrained is deprecated and will be removed in a future major release. Use TransformerBridge.boot_transformers(...) instead, then call enable_compatibility_mode() for HookedTransformer-equivalent numerics. See docs/source/content/migrating_to_v3.md.
  model = HookedTransformer.from_pretrained("meta-llama/Llama-3.2-1B-Instruct")
Loading weights: 100%|██████████| 146/146 [00:23<00:00,  6.13it/s]


Loaded pretrained model meta-llama/Llama-3.2-1B-Instruct into HookedTransformer


In [6]:
# Global variables
DATASET_FOLDER = str(REPO_ROOT / "datasets" / "abstractive")

MODEL_SHORT = "Llama-3.2-1B-Instruct"
D_NAME = "country-capital"
PATTERN_FOLDER = REPO_ROOT / "output" / MODEL_SHORT / D_NAME / "Heads" / "chunking" / "attn_pattern"
PATTERN_FOLDER.mkdir(parents=True, exist_ok=True)

## Loading prompts

In [7]:
dataset = _load_dataset(D_NAME, dataset_folder=DATASET_FOLDER)
filtered_dataset = filter_token_pairs_by_length(dataset, model=model, input_len=1, output_len=1)

In [11]:
prompts, answers, _ = create_few_shot_prompts(
    data=filtered_dataset,
    n_shot=5,
    delimiter=";", q_bos=" ", a_bos=" ", qa_delimiter=":")

In [12]:
# Randomely choose one prompt
prompt = prompts[43] 
prompt

' Syria: Damascus; Taiwan: Taipei; Thailand: Bangkok; Tunisia: Tunis; Turkey: Ankara; Ukraine:'

In [13]:
str_tokens = model.to_str_tokens(prompt)
str_tokens

['<|begin_of_text|>',
 ' Syria',
 ':',
 ' Damascus',
 ';',
 ' Taiwan',
 ':',
 ' Taipei',
 ';',
 ' Thailand',
 ':',
 ' Bangkok',
 ';',
 ' Tunisia',
 ':',
 ' Tunis',
 ';',
 ' Turkey',
 ':',
 ' Ankara',
 ';',
 ' Ukraine',
 ':']

## Attention patterns

In [14]:
def visualize_attn_pattern(prompt, layer, head):
    tokens = model.to_tokens(prompt)
    _, cache = model.run_with_cache(
        tokens, names_filter=lambda name: name == f"blocks.{layer}.attn.hook_pattern"
    )
    pattern = cache[f"blocks.{layer}.attn.hook_pattern"] # [1, n_heads, seq, seq]
    head_pattern = pattern[0, head] # [seq, seq] -- rows=query pos, cols=key pos

    ######### INTERACTIVE HTML ############
    fig = px.imshow(
        head_pattern.cpu().numpy(),
        x=str_tokens, y=str_tokens,
        labels={"x": "key position", "y": "query position", "color": "attention weight"},
        title=f"L{layer}H{head} attention pattern",
        color_continuous_scale="Greens",
    )

    fig.write_html(PATTERN_FOLDER / f"L{layer}H{head}_attn_pattern.html")

    ########## ARENA ATTN PRACTICE ############
    vis = cv_attn.attention_patterns(
        tokens=str_tokens,
        attention=pattern[0, head].unsqueeze(0),   # [1, seq, seq]
    )
    return vis


In [15]:
# L5H6
visualize_attn_pattern(prompt, layer=5, head=6)

In [16]:
# L6H6
visualize_attn_pattern(prompt, layer=6, head=6)

# Decode

In [ ]:
def decode_head_output(prompt, layer, head, top_k):
    tokens = model.to_tokens(prompt)   # [1, seq]
    _, cache = model.run_with_cache(
        tokens, names_filter=lambda name: name == f"blocks.{layer}.attn.hook_z"
    )
    z = cache[f"blocks.{layer}.attn.hook_z"]   # [1, seq, n_heads, d_head]
    z_h = z[0, :, head, :]   # [seq, d_head]
    head_out = z_h @ model.W_O[layer, head]   # [seq, d_model]
    logits = head_out @ model.W_U             # [seq, d_vocab] -- apply_ln=False, same as identify_heads.py
    topk_ids = logits.topk(top_k, dim=-1).indices   # [seq, k]
    decoded = [[model.tokenizer.decode([tid]) for tid in pos_ids.tolist()] for pos_ids in topk_ids]
    for pos, (tok, dec) in enumerate(zip(str_tokens, decoded)):
        if pos == 0:
            print(f"{pos}_bos_{tok}: {dec}")
        elif pos == logits.shape[1] - 2:
            print(f"{pos}_query_{tok}: {dec}")
        elif pos == logits.shape[1] - 1:
            print(f"{pos}_last_token_{tok}: {dec}")
        elif pos % 4 == 1:
            print(f"{pos}_input_{tok}: {dec}")
        elif pos % 4 == 3:
            print(f"{pos}_output_{tok}: {dec}")
        elif pos % 2 == 0:
            print(f"{pos}_separator_{tok}: {dec}")

In [ ]:
decode_head_output(prompt, layer=5, head=6, top_k=10)

0_bos_<|begin_of_text|>: ['Inf', 'angement', 'عل', 'etailed', 'ems', ' inf', ' Well', ' Lange', ' inflater', ' ob']
1_input_ Syria: ['Inf', 'angement', 'عل', 'etailed', 'ems', ' inf', ' Well', 'atio', ' Lange', ' ob']
2_separator_:: ['Inf', 'angement', 'etailed', ' inf', 'عل', 'atio', 'ems', ' Inf', 'peg', ' Well']
3_output_ Damascus: ['Inf', ' inf', 'atio', ' fingerprints', ' Achievement', 'ates', 'rowse', 'esor', 'ensitivity', ' Rath']
4_separator_;: ['Inf', ' inf', 'atio', ' Justice', 'Justice', ' fingerprints', ' Bridges', ' Inf', 'ates', ' Achievement']
5_input_ Taiwan: ['ister', 'isters', ' backward', 'erse', ' backwards', ' shack', 'atters', 'ering', 'manship', ' human']
6_separator_:: [' fingerprints', 'Justice', 'obe', ' Justice', 'ters', 'Inf', 'ates', ' backwards', 'ensitivity', ' inf']
7_output_ Taipei: ['isters', 'ister', 'manship', ' shack', 'erse', ' bench', 'znik', '-redux', 'nuts', ' backward']
8_separator_;: ['Inf', ' inf', 'Justice', 'atio', ' Justice', ' Inf', 'iden

# RSA

In [24]:
print("prompt:", repr(prompt))                    
print("len(str_tokens):", len(str_tokens))
print("expected len (4*n_shot+3):", 4*5 + 3)  
print(str_tokens)                                 

prompt: ' Syria: Damascus; Taiwan: Taipei; Thailand: Bangkok; Tunisia: Tunis; Turkey: Ankara; Ukraine:'
len(str_tokens): 23
expected len (4*n_shot+3): 23
['<|begin_of_text|>', ' Syria', ':', ' Damascus', ';', ' Taiwan', ':', ' Taipei', ';', ' Thailand', ':', ' Bangkok', ';', ' Tunisia', ':', ' Tunis', ';', ' Turkey', ':', ' Ankara', ';', ' Ukraine', ':']


In [25]:
import torch.nn.functional as F

def cosine_similarity_matrix(head_out: torch.Tensor) -> torch.Tensor:
    normed = F.normalize(head_out, dim=-1)   # [seq, d_model], each row unit norm
    return normed @ normed.T                  # [seq, seq], values in [-1, 1]

In [26]:
def build_rsa_matrix(prompt, layer, head):
    tokens = model.to_tokens(prompt)   # [1, seq]
    _, cache = model.run_with_cache(
        tokens, names_filter=lambda name: name == f"blocks.{layer}.attn.hook_z"
    )
    z = cache[f"blocks.{layer}.attn.hook_z"]   # [1, seq, n_heads, d_head]
    z_h = z[0, :, head, :]   # [seq, d_head]
    head_out = z_h @ model.W_O[layer, head]   # [seq, d_model]
    # head_out_centered = head_out - head_out.mean(dim=0, keepdim=True)
    # return cosine_similarity_matrix(head_out_centered)
    return cosine_similarity_matrix(head_out)

In [27]:
cos_sim_L5H6 = build_rsa_matrix(prompt, layer=5, head=6)
# cos_sim_L6H6 = build_rsa_matrix(prompt, layer=6, head=6)
cos_sim_L5H6.shape

torch.Size([23, 23])

In [28]:
# def plot_similarity(sim, str_tokens, title, save_path=None):
#     fig = px.imshow(
#         sim.detach().cpu().numpy(),
#         x=str_tokens, y=str_tokens,
#         labels={"color": "cosine similarity"},
#         title=title,
#         color_continuous_scale="Viridis", zmin=-1, zmax=1,
#     )
#     if save_path:
#         fig.write_html(save_path)
#     return fig

def plot_similarity(sim, str_tokens, title, save_path=None):
    unique_labels = [f"{i}_{tok}" for i, tok in enumerate(str_tokens)]
    fig = px.imshow(
        sim.detach().cpu().numpy(),
        x=unique_labels, y=unique_labels,
        labels={"color": "cosine similarity"},
        title=title,
        color_continuous_scale="Viridis", zmin=-1, zmax=1,
    )
    fig.update_xaxes(tickmode="linear", dtick=1, tickangle=90)
    fig.update_yaxes(tickmode="linear", dtick=1)
    fig.update_layout(width=900, height=900) 
    if save_path:
        save_path.parent.mkdir(parents=True, exist_ok=True)
        fig.write_html(save_path)
    return fig

plot_similarity(cos_sim_L5H6, str_tokens, "L5H6 representation similarity",
    save_path=PATTERN_FOLDER.parent / "rsa" / "L5H6_rsa.html")
# plot_similarity(cos_sim_L6H6, str_tokens, "L6H6 representation similarity",
#     save_path=PATTERN_FOLDER.parent / "rsa" / "L6H6_rsa.html")

In [29]:
for head in range(model.cfg.n_heads):
    cos_sim = build_rsa_matrix(prompt, layer=5, head=head)
    assert cos_sim.shape == torch.Size([23, 23]), "Wrong dimension for the axis 5 shots"
    plot_similarity(cos_sim, str_tokens, f"L5H{head} representation similarity",
        save_path=PATTERN_FOLDER.parent / "rsa" / f"L5H{head}_rsa.html")

In [30]:
for head in range(model.cfg.n_heads):
    cos_sim = build_rsa_matrix(prompt, layer=6, head=head)
    plot_similarity(cos_sim, str_tokens, f"L6H{head} representation similarity",
        save_path=PATTERN_FOLDER.parent / "rsa" / f"L6H{head}_rsa.html")